**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 16 — Interactive Demo (Gradio)

Four-tab Gradio interface:
1. **Predict** — upload image + text → label + confidence bar
2. **Explain** — GradCAM heatmap overlay
3. **Counterfactual** — swap tokens / occlude patches to flip prediction
4. **Human Review** — flag for HITL queue

All model logic inlined, no src/ imports. Runs publicly on Kaggle via `share=True`.

In [ ]:
import subprocess
for pkg in ["gradio>=4.0", "transformers", "torch", "torchvision"]:
    subprocess.run(["pip", "install", "-q", pkg], check=True)
print("Packages ready.")

In [ ]:
import os
import json
import re
import datetime
import io
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image, ImageDraw, ImageFont
from transformers import CLIPModel, CLIPProcessor, RobertaTokenizer, RobertaForMaskedLM
import gradio as gr

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Device         : {DEVICE}")

In [ ]:
# â”€â”€ Model definitions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self,pv,ids,mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv),-1),
                F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,d=512,h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self,i,t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,pv,ids,mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta

# GradCAM hook
class CLIPGradCAM:
    def __init__(self, model):
        self.model=model; self.grads=[]; self.acts=[]
        layer=model.encoder.clip.vision_model.encoder.layers[-1]
        layer.register_forward_hook(lambda m,i,o: self.acts.append(o[0].detach()))
        layer.register_full_backward_hook(lambda m,gi,go: self.grads.append(go[0].detach()))
    def compute(self,pv,ids,mask):
        self.grads.clear(); self.acts.clear()
        pv.requires_grad_(True)
        logits,_,_=self.model(pv,ids,mask)
        self.model.zero_grad()
        logits[0,1].backward()
        act=self.acts[0]; grd=self.grads[0]
        w=grd.mean(dim=[0,1])
        cam=(act*w.unsqueeze(0).unsqueeze(0)).sum(-1).squeeze(0)
        cam=cam[1:]  # drop CLS
        h=w_=int(cam.shape[0]**0.5)
        cam=cam.reshape(h,w_).numpy(); cam=np.maximum(cam,0)
        if cam.max()>0: cam/=cam.max()
        return cam

# Load everything
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = HatefulMemeClassifier().to(DEVICE)
ckpt      = os.path.join(OUTPUT_DIR, "cross_attention_best.pt")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("Model loaded.")
else:
    print("WARNING: running with random weights.")
model.eval()
gradcam = CLIPGradCAM(model)

# RoBERTa for text CFs
roberta_tok = RobertaTokenizer.from_pretrained("roberta-base")
roberta_mlm = RobertaForMaskedLM.from_pretrained("roberta-base").to(DEVICE).eval()

In [ ]:
# â”€â”€ Helper functions â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def clean_text(text):
    if not isinstance(text, str): return "[no text]"
    return re.sub(r"\s+"," ", re.sub(r"@\w+"," ", re.sub(r"https?://\S+"," ",text))).strip() or "[no text]"

@torch.no_grad()
def predict(pil_img, text, threshold=0.5):
    txt = clean_text(text)
    enc = processor(text=[txt], images=pil_img.convert("RGB"),
                    return_tensors="pt", padding="max_length", max_length=77, truncation=True)
    pv   = enc["pixel_values"].to(DEVICE)
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)
    logits,_,_ = model(pv, ids, mask)
    probs = torch.softmax(logits, -1)[0].cpu()
    return float(probs[0]), float(probs[1])

def make_gradcam_overlay(pil_img, text, alpha=0.45):
    txt = clean_text(text)
    img = pil_img.convert("RGB").resize((224,224))
    enc = processor(text=[txt], images=img, return_tensors="pt",
                    padding="max_length", max_length=77, truncation=True)
    pv   = enc["pixel_values"].to(DEVICE)
    ids  = enc["input_ids"].to(DEVICE)
    mask = enc["attention_mask"].to(DEVICE)

    pv.requires_grad_(False)
    cam = gradcam.compute(pv.clone().detach().requires_grad_(True), ids, mask)

    cam_resized = Image.fromarray((cm.jet(cam)[:,:,:3]*255).astype(np.uint8)).resize((224,224))
    overlay = Image.blend(img, cam_resized, alpha)
    return overlay

def text_counterfactual_demo(pil_img, text, threshold=0.5):
    txt    = clean_text(text)
    orig_p = predict(pil_img, text)[1]
    words  = txt.split()
    result_list = []

    for i in range(min(len(words), 15)):
        masked = words.copy(); masked[i] = "<mask>"
        masked_txt = " ".join(masked)
        tokenized  = roberta_tok(masked_txt, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = roberta_mlm(**tokenized).logits
        mask_pos = (tokenized["input_ids"][0] == roberta_tok.mask_token_id).nonzero().item()
        top5_ids = out[0, mask_pos].topk(5).indices.tolist()

        for tok_id in top5_ids:
            sub = roberta_tok.decode([tok_id]).strip()
            if sub.lower() == words[i].lower(): continue
            new_words = words.copy(); new_words[i] = sub
            new_text  = " ".join(new_words)
            new_p     = predict(pil_img, new_text)[1]
            flipped   = (orig_p >= threshold) != (new_p >= threshold)
            result_list.append(f"  '{words[i]}' â†’ '{sub}' | P(hate): {orig_p:.2f} â†’ {new_p:.2f}{'  âœ¦ FLIP' if flipped else ''}")
            if flipped: break

    return "\n".join(result_list) if result_list else "No substitution effects found."

In [ ]:
# â”€â”€ Gradio app â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
AUDIT_PATH = os.path.join(OUTPUT_DIR, "hitl_audit_log.jsonl")

def tab_predict(image, text, threshold):
    if image is None:
        return "Please upload an image.", {"non_hateful": 0.5, "hateful": 0.5}
    p_non, p_hate = predict(image, text, threshold)
    label = "HATEFUL" if p_hate >= threshold else "NON-HATEFUL"
    msg = f"**{label}** (confidence: {max(p_hate, p_non):.1%})"
    return msg, {"non_hateful": p_non, "hateful": p_hate}

def tab_explain(image, text):
    if image is None:
        return None
    try:
        model.train()  # needed for grad
        overlay = make_gradcam_overlay(image, text)
        model.eval()
        return overlay
    except Exception as e:
        model.eval()
        return None

def tab_counterfactual(image, text, threshold):
    if image is None:
        return "Upload an image first."
    p_non, p_hate = predict(image, text, threshold)
    orig_msg = f"Original: P(hate)={p_hate:.3f} â†’ {'HATEFUL' if p_hate >= threshold else 'NON-HATEFUL'}\n\n"
    cf_report = text_counterfactual_demo(image, text, threshold)
    return orig_msg + "Text counterfactuals (token substitution):\n" + cf_report

def tab_flag(image, text, annotator_note):
    if image is None:
        return "Upload an image first."
    p_non, p_hate = predict(image, text)
    entry = {
        "id": f"demo_{datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S%f')}",
        "source": "gradio_demo",
        "text": text,
        "p_hateful": p_hate,
        "model_pred": 1 if p_hate >= 0.5 else 0,
        "human_label": -1,  # flagged for review
        "annotator_note": annotator_note,
        "timestamp": datetime.datetime.utcnow().isoformat(),
    }
    with open(AUDIT_PATH, "a") as f:
        f.write(json.dumps(entry) + "\n")
    return f"Flagged for review. Model confidence P(hate)={p_hate:.3f}. Entry saved."

with gr.Blocks(title="Hateful Meme Detector", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# Hateful Meme Detector\nCLIP + Cross-Attention multimodal classifier")

    with gr.Tabs():
        # Tab 1: Prediction
        with gr.TabItem("Predict"):
            gr.Markdown("Upload a meme image and provide its text to get a prediction.")
            with gr.Row():
                img_in   = gr.Image(type="pil", label="Meme Image")
                with gr.Column():
                    txt_in   = gr.Textbox(label="Meme Text", lines=3)
                    thresh   = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Classification Threshold")
                    pred_btn = gr.Button("Classify", variant="primary")
            pred_out   = gr.Markdown()
            conf_bar   = gr.Label(label="Class Probabilities")
            pred_btn.click(fn=tab_predict, inputs=[img_in, txt_in, thresh], outputs=[pred_out, conf_bar])

        # Tab 2: Explanation
        with gr.TabItem("Explain (GradCAM)"):
            gr.Markdown("Visualise which image regions most influenced the model's decision.")
            with gr.Row():
                exp_img  = gr.Image(type="pil", label="Meme Image")
                exp_txt  = gr.Textbox(label="Meme Text")
            exp_btn  = gr.Button("Generate Explanation", variant="primary")
            exp_out  = gr.Image(label="GradCAM Heatmap Overlay")
            exp_btn.click(fn=tab_explain, inputs=[exp_img, exp_txt], outputs=[exp_out])

        # Tab 3: Counterfactual
        with gr.TabItem("Counterfactual"):
            gr.Markdown("Find minimal text changes that would flip the model's prediction.")
            with gr.Row():
                cf_img   = gr.Image(type="pil", label="Meme Image")
                with gr.Column():
                    cf_txt   = gr.Textbox(label="Meme Text", lines=3)
                    cf_thr   = gr.Slider(0.1, 0.9, value=0.5, step=0.05, label="Threshold")
                    cf_btn   = gr.Button("Generate Counterfactuals", variant="primary")
            cf_out = gr.Textbox(label="Counterfactual Analysis", lines=12)
            cf_btn.click(fn=tab_counterfactual, inputs=[cf_img, cf_txt, cf_thr], outputs=[cf_out])

        # Tab 4: Human Review
        with gr.TabItem("Flag for Review"):
            gr.Markdown("Flag uncertain or edge-case memes for human review.")
            with gr.Row():
                flag_img  = gr.Image(type="pil", label="Meme Image")
                with gr.Column():
                    flag_txt  = gr.Textbox(label="Meme Text", lines=3)
                    flag_note = gr.Textbox(label="Review Note (optional)", lines=2)
                    flag_btn  = gr.Button("Flag for Human Review", variant="stop")
            flag_out = gr.Textbox(label="Status")
            flag_btn.click(fn=tab_flag, inputs=[flag_img, flag_txt, flag_note], outputs=[flag_out])

print("Gradio app built. Launching...")

In [ ]:
# â”€â”€ Launch â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
# share=True creates a public URL accessible outside Kaggle
demo.launch(share=True, quiet=False)